# Crypto research — Experiment 001 cloud reproduction

All training must run in hosted Kaggle or Colab, never on the laptop. Keep this notebook private and enable Internet for public archive downloads. No exchange accounts or order APIs. Cloud CPU is sufficient. This notebook has not been cloud-executed. The earlier local result is historical and separate. Reproducing it in the cloud does not create a fresh holdout or demonstrate better performance. Provider quotas and access depend on your account. This downloads 192 small monthly archives plus checksum files. Do not publish raw data or model artifacts as commercially licensed data.

# Experiment 001 — frozen before evaluation

Created 2026-09-20. Scope: private numerical feasibility study, not market-wide coin discovery.

- Universe fixed to BTCUSDT and ETHUSDT; daily Binance spot candles, 2018-01-01 through 2025-12-31. This survivorship-biased pair tests infrastructure and prediction only.
- Features known after day t closes: returns over 1/3/7/14/30 days, 7/30-day volatility, 20/60-day mean-price gaps, daily high-low range, candle body, volume relative to trailing 30-day mean, 30-day drawdown, and asset indicator.
- Decision after completed day t. Simulated entry at day t+2 open, exit t+3 open (a full-day delay avoids assuming instant execution at the closing boundary). Label is positive entry-to-exit return. Thus the prediction horizon is 24 hours after entry, not immediately after the observation. Purge labels whose exit touches the next split.
- Development walk-forward: train from 2018 through data with exits before 2022, validate 2022; expand training and validate 2023. No random shuffle. Validation folds pooled for model selection.
- Models: StandardScaler + LogisticRegression(C=1, max_iter=1000); HistGradientBoostingClassifier(max_iter=100, max_leaf_nodes=7, learning_rate=0.05, l2_regularization=1, min_samples_leaf=40, early_stopping=False). Seed 42. No hyperparameter sweep.
- Pick model using mean validation Brier score (lower is better); tie favors logistic regression. Retrain on observations with labels ending before 2024. Freeze model for untouched 2024-2025 test. No probability calibration fit on test.
- Long when P(up) >= 0.55, otherwise cash. No shorts, leverage, stops or intraday fill claims. Threshold fixed before evaluation. Daily equal-weight two-sleeve portfolio; each sleeve 50% capital, unused capital cash, zero cash yield.
- Each active one-day trade pays 15 basis points per side (10 fee + 5 spread/slippage), 30 basis points round trip; stress at 60 basis points round trip. These are assumptions, not verified account fees. Close/reopen every active daily holding, including consecutive active days; conservative cost convention shared by daily strategies. Buy-and-hold pays entry/exit only and drifts in weight.
- Baselines: cash, two-asset buy-and-hold, 20/60-day long/cash trend, expanding-training prevalence for probabilities. Report turnover/trade count and exposure so cost differences are visible.
- Metrics: Brier score, log loss, accuracy, net cumulative return, daily Sharpe (365-day annualization, descriptive), maximum drawdown, exposure, active asset-days, by asset/year and causal bull/bear trend regime. Use paired 14-day block bootstrap, 2,000 replicates, for mean daily return difference versus trend; correlated days and both assets must stay together.
- Preliminary numerical pass requires: test Brier lower than training-prevalence baseline; positive test net return at both costs; lower 95% block-bootstrap bound for excess return over trend > 0; >=100 active asset-days; nonnegative net returns in both test years; maximum drawdown <=25%. These are engineering research gates, not a profitability guarantee.
- Part 3 remains blocked even on numerical pass until source-grounding tests and point-in-time discovery/liquidity checks pass. A failure is reported without retuning on test. Any next experiment gets a new ID and needs a new untouched/prospective evaluation period.
- Reserve 2026 data for future forward evaluation; do not inspect it for this experiment. Prospective paper evidence needs at least 90 calendar days and 100 resolved assessments, with predeclared costs and risk limits; elapsed time cannot be accelerated by backtesting.


In [ ]:
import os, platform
from pathlib import Path
kaggle = platform.system() == "Linux" and bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) and Path("/kaggle/working").is_dir()
colab = platform.system() == "Linux" and bool(os.environ.get("COLAB_RELEASE_TAG")) and Path("/content").is_dir()
if not (kaggle or colab):
    raise RuntimeError("Cloud-only policy: use hosted Kaggle or Colab, never a local runtime.")
print("Hosted runtime:", "Kaggle" if kaggle else "Colab")


In [ ]:
%pip install numpy==1.26.4 pandas==2.2.2 scikit-learn==1.7.2 requests==2.34.2 joblib==1.5.3 threadpoolctl==3.6.0


In [ ]:
from pathlib import Path
import os
project = Path("/content/crypto-research") if Path("/content").exists() else Path.cwd()/"crypto-research"
(project/"scripts").mkdir(parents=True, exist_ok=True)
(project/"research").mkdir(exist_ok=True)
os.chdir(project)
(project/"scripts"/"experiment.py").write_text('"""Experiment 001: private, daily spot research. No order or account APIs."""\nimport argparse\nimport calendar\nimport concurrent.futures\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport time\nimport zipfile\nfrom datetime import datetime, timezone\n\nos.environ.setdefault(\'OMP_NUM_THREADS\', \'2\')\nos.environ.setdefault(\'OPENBLAS_NUM_THREADS\', \'2\')\nimport numpy as np\nimport pandas as pd\nimport requests\nimport sklearn\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import brier_score_loss, log_loss, accuracy_score\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\n\nROOT = Path(__file__).resolve().parents[1]\nDATA = ROOT / \'data\'\nOUT = ROOT / \'results\' / \'experiment_001\'\nSYMBOLS = [\'BTCUSDT\', \'ETHUSDT\']\nCOLS = [\'timestamp\',\'open\',\'high\',\'low\',\'close\',\'volume\',\'close_time\',\'quote_volume\',\n        \'trades\',\'taker_base\',\'taker_quote\',\'ignore\']\nFEATURES = [\'r1\',\'r3\',\'r7\',\'r14\',\'r30\',\'vol7\',\'vol30\',\'gap20\',\'gap60\',\n            \'range\',\'body\',\'volume_ratio\',\'drawdown30\',\'asset_eth\']\n\ndef utcnow():\n    return datetime.now(timezone.utc).isoformat()\n\ndef digest(b):\n    return hashlib.sha256(b).hexdigest()\n\ndef require_cloud_training():\n    """Fail closed outside recognized hosted runtimes; no local override."""\n    linux = platform.system() == \'Linux\'\n    kaggle = linux and bool(os.environ.get(\'KAGGLE_KERNEL_RUN_TYPE\')) and Path(\'/kaggle/working\').is_dir()\n    colab = linux and bool(os.environ.get(\'COLAB_RELEASE_TAG\')) and Path(\'/content\').is_dir()\n    if kaggle:\n        return \'Kaggle CPU\'\n    if colab:\n        return \'Colab CPU\'\n    raise RuntimeError(\'Cloud-only training policy: run in a hosted Kaggle or Colab runtime. Laptop training is disabled.\')\n\ndef fetch_month(task):\n    symbol, year, month = task\n    name = f\'{symbol}-1d-{year}-{month:02d}.zip\'\n    url = f\'https://data.binance.vision/data/spot/monthly/klines/{symbol}/1d/{name}\'\n    path = DATA / \'raw\' / name\n    meta_path = path.with_suffix(\'.json\')\n    if path.exists() and meta_path.exists():\n        meta = json.loads(meta_path.read_text())\n        if digest(path.read_bytes()) == meta[\'sha256\'] == meta[\'provider_sha256\']:\n            return meta\n        raise ValueError(f\'Cached archive checksum mismatch: {name}\')\n    for attempt in range(3):\n        try:\n            r = requests.get(url, timeout=30)\n            r.raise_for_status()\n            check = requests.get(url + \'.CHECKSUM\', timeout=30)\n            check.raise_for_status()\n            expected = check.text.split()[0].lower()\n            actual = digest(r.content)\n            if actual != expected:\n                raise ValueError(f\'Provider checksum mismatch: {name}\')\n            meta = dict(symbol=symbol, year=year, month=month, url=url,\n                        retrieved_at=utcnow(), sha256=actual, provider_sha256=expected,\n                        bytes=len(r.content), parser_version=\'1\',\n                        use=\'private backtesting only; redistribution not cleared\')\n            path.write_bytes(r.content)\n            meta_path.write_text(json.dumps(meta, indent=2), encoding=\'utf-8\')\n            return meta\n        except requests.RequestException:\n            if attempt == 2:\n                raise\n            time.sleep(1 + attempt)\n\ndef parse_archive(path):\n    with zipfile.ZipFile(path) as z:\n        files = [n for n in z.namelist() if n.endswith(\'.csv\')]\n        if len(files) != 1:\n            raise ValueError(f\'Unexpected archive contents: {path}\')\n        frame = pd.read_csv(io.BytesIO(z.read(files[0])), names=COLS)\n    raw = pd.to_numeric(frame.timestamp, errors=\'raise\')\n    # Preserve integer precision, independently normalize every observation.\n    millis = raw.where(raw < 10**14, raw // 1000)\n    frame[\'date\'] = pd.to_datetime(millis, unit=\'ms\', utc=True).dt.tz_localize(None)\n    for col in COLS[1:]:\n        frame[col] = pd.to_numeric(frame[col], errors=\'raise\')\n    return frame\n\ndef audit(frame):\n    if frame.date.duplicated().any() or not frame.date.is_monotonic_increasing:\n        raise ValueError(\'Dates duplicated or unsorted\')\n    if not (frame.date.diff().dropna() == pd.Timedelta(days=1)).all():\n        raise ValueError(\'Missing daily candles\')\n    if not (frame.date == frame.date.dt.normalize()).all():\n        raise ValueError(\'Non-midnight daily candle\')\n    prices = frame[[\'open\',\'high\',\'low\',\'close\']]\n    if not np.isfinite(prices.to_numpy()).all() or (prices <= 0).any().any():\n        raise ValueError(\'Invalid prices\')\n    if (frame.high < prices.max(axis=1)).any() or (frame.low > prices.min(axis=1)).any():\n        raise ValueError(\'Impossible OHLC\')\n    if not np.isfinite(frame.volume).all() or (frame.volume < 0).any():\n        raise ValueError(\'Invalid volume\')\n\ndef prepare():\n    (DATA / \'raw\').mkdir(parents=True, exist_ok=True)\n    OUT.mkdir(parents=True, exist_ok=True)\n    tasks = [(s,y,m) for s in SYMBOLS for y in range(2018,2026) for m in range(1,13)]\n    records = []\n    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as pool:\n        for i, meta in enumerate(pool.map(fetch_month, tasks), 1):\n            records.append(meta)\n            if i % 24 == 0:\n                print(f\'Verified archives: {i}/{len(tasks)}\', flush=True)\n    frames = []\n    for symbol in SYMBOLS:\n        parts = []\n        for s, y, m in tasks:\n            if s != symbol:\n                continue\n            f = parse_archive(DATA / \'raw\' / f\'{s}-1d-{y}-{m:02d}.zip\')\n            if len(f) != calendar.monthrange(y,m)[1] or not ((f.date.dt.year == y) & (f.date.dt.month == m)).all():\n                raise ValueError(\'Archive month coverage mismatch\')\n            parts.append(f)\n        frame = pd.concat(parts, ignore_index=True).sort_values(\'date\')\n        audit(frame)\n        frame[\'symbol\'] = symbol\n        frames.append(frame)\n    pd.concat(frames, ignore_index=True).to_csv(DATA / \'candles.csv\', index=False)\n    manifest = dict(created_at=utcnow(), archives=records,\n                    candles_sha256=digest((DATA/\'candles.csv\').read_bytes()),\n                    rows=sum(map(len, frames)),\n                    rights_evidence=[\'https://github.com/binance/binance-public-data\',\n                    \'https://www.binance.com/en/blog/futures/421499824684901131\'])\n    (OUT/\'data_manifest.json\').write_text(json.dumps(manifest, indent=2))\n    print(f\'Data audit passed: {manifest["rows"]} candles\', flush=True)\n\ndef features(f):\n    f = f.sort_values(\'date\').copy()\n    for n in [1,3,7,14,30]:\n        f[f\'r{n}\'] = f.close.pct_change(n, fill_method=None)\n    for n in [7,30]:\n        f[f\'vol{n}\'] = f.r1.rolling(n).std()\n    for n in [20,60]:\n        f[f\'gap{n}\'] = f.close / f.close.rolling(n).mean() - 1\n    f[\'range\'] = (f.high-f.low)/f.close\n    f[\'body\'] = f.close/f.open-1\n    f[\'volume_ratio\'] = f.volume/f.volume.rolling(30).mean().replace(0,np.nan)\n    f[\'drawdown30\'] = f.close/f.close.rolling(30).max()-1\n    f[\'asset_eth\'] = (f.symbol == \'ETHUSDT\').astype(int)\n    f[\'entry\'] = f.date.shift(-2)\n    f[\'exit\'] = f.date.shift(-3)\n    f[\'future_return\'] = f.open.shift(-3)/f.open.shift(-2)-1\n    f[\'up\'] = (f.future_return > 0).astype(int)\n    f[\'trend\'] = (f.close.rolling(20).mean() > f.close.rolling(60).mean()).astype(int)\n    return f\n\ndef new_models():\n    return dict(logistic=make_pipeline(StandardScaler(), LogisticRegression(C=1,max_iter=1000,random_state=42)),\n                histogram=HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=7,\n                    learning_rate=0.05,l2_regularization=1,min_samples_leaf=40,\n                    early_stopping=False,random_state=42))\n\ndef score(y,p):\n    return dict(brier=float(brier_score_loss(y,p)), log_loss=float(log_loss(y,p,labels=[0,1])),\n                accuracy=float(accuracy_score(y,p >= .5)), n=len(y))\n\ndef performance(r):\n    r = np.asarray(r, dtype=float)\n    equity = np.r_[1.,np.cumprod(1+r)]\n    dd = equity/np.maximum.accumulate(equity)-1\n    return dict(net_return=float(equity[-1]-1), max_drawdown=float(-dd.min()),\n                daily_sharpe=float(np.mean(r)/np.std(r,ddof=1)*np.sqrt(365)) if len(r)>1 and np.std(r)>0 else 0.,\n                days=len(r))\n\ndef net_returns(frame, active, cost):\n    # Costs on each leg: buy at marked-up entry and sell at marked-down exit.\n    return np.asarray(active) * ((1+frame.future_return.to_numpy())*(1-cost)/(1+cost)-1)\n\ndef bootstrap_difference(a,b):\n    d=np.asarray(a)-np.asarray(b)\n    rng=np.random.default_rng(42)\n    means=[]\n    for _ in range(2000):\n        starts=rng.integers(0,len(d),size=int(np.ceil(len(d)/14)))\n        idx=np.concatenate([(s+np.arange(14)) % len(d) for s in starts])[:len(d)]\n        means.append(d[idx].mean())\n    return dict(mean_daily_excess=float(d.mean()),ci95=list(map(float,np.quantile(means,[.025,.975]))),\n                method=\'paired circular 14-day blocks; 2000 replicates; both assets grouped by date\')\n\ndef evaluate():\n    environment = require_cloud_training()\n    started=time.perf_counter()\n    manifest=json.loads((OUT/\'data_manifest.json\').read_text())\n    if digest((DATA/\'candles.csv\').read_bytes()) != manifest[\'candles_sha256\']:\n        raise ValueError(\'Processed data changed since manifest\')\n    raw=pd.read_csv(DATA/\'candles.csv\',parse_dates=[\'date\'])\n    frame=pd.concat([features(g) for _,g in raw.groupby(\'symbol\')],ignore_index=True)\n    frame=frame.replace([np.inf,-np.inf],np.nan).dropna(subset=FEATURES+[\'future_return\',\'entry\',\'exit\'])\n    validation=[]\n    with threadpool_limits(limits=2):\n        for year in [2022,2023]:\n            boundary=pd.Timestamp(f\'{year}-01-01\')\n            train=frame[frame.exit < boundary]\n            valid=frame[(frame.entry >= boundary)&(frame.exit < pd.Timestamp(f\'{year+1}-01-01\'))]\n            for name,model in new_models().items():\n                model.fit(train[FEATURES],train.up)\n                p=model.predict_proba(valid[FEATURES])[:,1]\n                validation.append(dict(year=year,model=name,**score(valid.up,p)))\n        v=pd.DataFrame(validation)\n        means=v.groupby(\'model\').brier.mean()\n        selected=\'logistic\' if means[\'logistic\'] <= means[\'histogram\'] else \'histogram\'\n        train=frame[frame.exit < pd.Timestamp(\'2024-01-01\')]\n        test=frame[(frame.entry >= pd.Timestamp(\'2024-01-01\')) & (frame.exit < pd.Timestamp(\'2026-01-01\'))].copy()\n        model=new_models()[selected]\n        model.fit(train[FEATURES],train.up)\n        test[\'probability\']=model.predict_proba(test[FEATURES])[:,1]\n    test[\'active\']=(test.probability >= .55).astype(int)\n    test[\'regime\']=np.where(test.trend == 1,\'bull_trend\',\'bear_trend\')\n    prevalence=float(train.up.mean())\n    result=dict(experiment=\'001\',executed_at=utcnow(),execution_environment=environment,\n                platform=platform.platform(),python=platform.python_version(),\n                versions=dict(numpy=np.__version__,pandas=pd.__version__,sklearn=sklearn.__version__),\n                features=FEATURES,selected_model=selected,validation=validation,\n                test_prediction=score(test.up,test.probability),\n                prevalence_baseline=score(test.up,np.full(len(test),prevalence)),\n                training_rows=len(train),test_rows=len(test),\n                first_entry=str(test.entry.min()),last_exit=str(test.exit.max()),\n                active_asset_days=int(test.active.sum()),exposure=float(test.active.mean()),\n                costs={},by_asset={},by_year={},by_regime={})\n    daily_all=None\n    for cost in [.0015,.003]:\n        key=f\'{cost*20000:.0f}bps_round_trip\'\n        daily={}\n        for name,active in [(\'model\',test.active),(\'trend\',test.trend),(\'cash\',np.zeros(len(test)))]:\n            test[f\'{name}_net\']=net_returns(test,active,cost)\n            # Each daily sleeve is 50% of total equity, cash for inactive sleeve.\n            daily[name]=test.groupby(\'entry\')[f\'{name}_net\'].mean().sort_index()\n        matrix=test.pivot(index=\'entry\',columns=\'symbol\',values=\'future_return\').sort_index()\n        if matrix.isna().any().any() or matrix.shape[1] != 2:\n            raise ValueError(\'Unequal test coverage\')\n        wealth=(1+matrix).cumprod()/(1+cost)\n        wealth.iloc[-1] *= (1-cost)\n        portfolio=wealth.mean(axis=1)\n        daily[\'buy_hold\']=portfolio.pct_change(fill_method=None)\n        daily[\'buy_hold\'].iloc[0]=portfolio.iloc[0]-1\n        result[\'costs\'][key]={name:performance(r) for name,r in daily.items()}\n        result[\'costs\'][key][\'model_vs_trend\']=bootstrap_difference(daily[\'model\'],daily[\'trend\'])\n        if cost == .0015:\n            daily_all=pd.DataFrame(daily)\n            for symbol,g in test.groupby(\'symbol\'):\n                result[\'by_asset\'][symbol]=dict(**performance(g.model_net),active_asset_days=int(g.active.sum()))\n            for year,g in daily_all.groupby(daily_all.index.year):\n                result[\'by_year\'][str(year)]={name:performance(g[name]) for name in daily_all.columns}\n            for regime,g in test.groupby(\'regime\'):\n                result[\'by_regime\'][regime]=dict(n=len(g),active_asset_days=int(g.active.sum()),\n                    mean_net_return=float(g.model_net.mean()),prediction=score(g.up,g.probability))\n            test.to_csv(OUT/\'test_predictions.csv\',index=False)\n    base=result[\'costs\'][\'30bps_round_trip\']\n    gates=dict(brier_improves=result[\'test_prediction\'][\'brier\'] < result[\'prevalence_baseline\'][\'brier\'],\n               positive_net=base[\'model\'][\'net_return\'] > 0,\n               positive_stress=result[\'costs\'][\'60bps_round_trip\'][\'model\'][\'net_return\'] > 0,\n               excess_ci_positive=base[\'model_vs_trend\'][\'ci95\'][0] > 0,\n               sufficient_trades=result[\'active_asset_days\'] >=100,\n               both_years_nonnegative=all(x[\'model\'][\'net_return\']>=0 for x in result[\'by_year\'].values()),\n               drawdown_limit=base[\'model\'][\'max_drawdown\'] <= .25)\n    result[\'gates\']=gates\n    result[\'numerical_gate\']=\'PASS\' if all(gates.values()) else \'FAIL\'\n    result[\'part3_gate\']=\'NOT_READY: discovery, liquidity and explanation evaluation remain unproven\'\n    result[\'wall_seconds\']=time.perf_counter()-started\n    result[\'code_sha256\']=digest(Path(__file__).read_bytes())\n    result[\'protocol_sha256\']=digest((ROOT/\'research\'/\'EVALUATION_PROTOCOL.md\').read_bytes())\n    result[\'candles_sha256\']=manifest[\'candles_sha256\']\n    (OUT/\'metrics.json\').write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n    daily_all.to_csv(OUT/\'daily_portfolio_returns.csv\')\n    import joblib\n    joblib.dump(model,OUT/\'numerical_model.joblib\')\n    print(json.dumps(result,indent=2),flush=True)\n\nif __name__ == \'__main__\':\n    parser=argparse.ArgumentParser()\n    parser.add_argument(\'action\',choices=[\'prepare\',\'evaluate\',\'all\'])\n    args=parser.parse_args()\n    if args.action in [\'evaluate\',\'all\']:\n        require_cloud_training()\n    if args.action in [\'prepare\',\'all\']:\n        prepare()\n    if args.action in [\'evaluate\',\'all\']:\n        evaluate()\n', encoding="utf-8")
(project/"research"/"EVALUATION_PROTOCOL.md").write_text('# Experiment 001 — frozen before evaluation\n\nCreated 2026-09-20. Scope: private numerical feasibility study, not market-wide coin discovery.\n\n- Universe fixed to BTCUSDT and ETHUSDT; daily Binance spot candles, 2018-01-01 through 2025-12-31. This survivorship-biased pair tests infrastructure and prediction only.\n- Features known after day t closes: returns over 1/3/7/14/30 days, 7/30-day volatility, 20/60-day mean-price gaps, daily high-low range, candle body, volume relative to trailing 30-day mean, 30-day drawdown, and asset indicator.\n- Decision after completed day t. Simulated entry at day t+2 open, exit t+3 open (a full-day delay avoids assuming instant execution at the closing boundary). Label is positive entry-to-exit return. Thus the prediction horizon is 24 hours after entry, not immediately after the observation. Purge labels whose exit touches the next split.\n- Development walk-forward: train from 2018 through data with exits before 2022, validate 2022; expand training and validate 2023. No random shuffle. Validation folds pooled for model selection.\n- Models: StandardScaler + LogisticRegression(C=1, max_iter=1000); HistGradientBoostingClassifier(max_iter=100, max_leaf_nodes=7, learning_rate=0.05, l2_regularization=1, min_samples_leaf=40, early_stopping=False). Seed 42. No hyperparameter sweep.\n- Pick model using mean validation Brier score (lower is better); tie favors logistic regression. Retrain on observations with labels ending before 2024. Freeze model for untouched 2024-2025 test. No probability calibration fit on test.\n- Long when P(up) >= 0.55, otherwise cash. No shorts, leverage, stops or intraday fill claims. Threshold fixed before evaluation. Daily equal-weight two-sleeve portfolio; each sleeve 50% capital, unused capital cash, zero cash yield.\n- Each active one-day trade pays 15 basis points per side (10 fee + 5 spread/slippage), 30 basis points round trip; stress at 60 basis points round trip. These are assumptions, not verified account fees. Close/reopen every active daily holding, including consecutive active days; conservative cost convention shared by daily strategies. Buy-and-hold pays entry/exit only and drifts in weight.\n- Baselines: cash, two-asset buy-and-hold, 20/60-day long/cash trend, expanding-training prevalence for probabilities. Report turnover/trade count and exposure so cost differences are visible.\n- Metrics: Brier score, log loss, accuracy, net cumulative return, daily Sharpe (365-day annualization, descriptive), maximum drawdown, exposure, active asset-days, by asset/year and causal bull/bear trend regime. Use paired 14-day block bootstrap, 2,000 replicates, for mean daily return difference versus trend; correlated days and both assets must stay together.\n- Preliminary numerical pass requires: test Brier lower than training-prevalence baseline; positive test net return at both costs; lower 95% block-bootstrap bound for excess return over trend > 0; >=100 active asset-days; nonnegative net returns in both test years; maximum drawdown <=25%. These are engineering research gates, not a profitability guarantee.\n- Part 3 remains blocked even on numerical pass until source-grounding tests and point-in-time discovery/liquidity checks pass. A failure is reported without retuning on test. Any next experiment gets a new ID and needs a new untouched/prospective evaluation period.\n- Reserve 2026 data for future forward evaluation; do not inspect it for this experiment. Prospective paper evidence needs at least 90 calendar days and 100 resolved assessments, with predeclared costs and risk limits; elapsed time cannot be accelerated by backtesting.\n', encoding="utf-8")


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/experiment.py", "all"], check=True)


In [ ]:
import shutil
archive = shutil.make_archive(str(project/"experiment_001_results"), "zip", project/"results")
print(archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Save the results archive from the notebook file browser before the runtime ends.")
